#AGENT MÁY HÚT BỤI BẰNG BFS, DFS

In [78]:
import random
from collections import deque

class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

def get_actions(state):
    """Lấy các hành động hợp lệ từ trạng thái hiện tại"""
    x, y, grid = state
    row, col = len(grid), len(grid[0])
    actions = []

    # NẾU CÓ RÁC -> BẮT BUỘC CHỈ CÓ 1 HÀNH ĐỘNG LÀ DỌN DẸP
    if grid[x][y] == 1:
        return ["CLEAN"]  # Bỏ luôn các hướng đi khác

    # NẾU KHÔNG CÓ RÁC -> MỚI ĐƯỢC PHÉP DI CHUYỂN
    if x > 0 and grid[x-1][y] != 2: actions.append("UP")
    if x < row-1 and grid[x+1][y] != 2: actions.append("DOWN")
    if y > 0 and grid[x][y-1] != 2: actions.append("LEFT")
    if y < col-1 and grid[x][y+1] != 2: actions.append("RIGHT")

    return actions

def get_child_state(state, action):
    """Tính toán trạng thái mới sau khi thực hiện hành động"""
    x, y, grid = state
    if action == "CLEAN":
        new_grid = list(list(r) for r in grid)
        new_grid[x][y] = 0
        return (x, y, tuple(tuple(r) for r in new_grid))
    elif action == "UP":    return (x-1, y, grid)
    elif action == "DOWN":  return (x+1, y, grid)
    elif action == "LEFT":  return (x, y-1, grid)
    elif action == "RIGHT": return (x, y+1, grid)

def goal_test(state):
    """Kiểm tra xem sàn nhà đã sạch hết chưa (không còn số 1)"""
    _, _, grid = state
    for row in grid:
        if 1 in row:
            return False
    return True

def get_solution(node):
    """Truy xuất đường đi từ gốc đến đích"""
    path = []
    while node.parent is not None:
        path.append(node.action)
        node = node.parent
    return path[::-1]

In [79]:
def BFS_Kieu_1(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    reached = set()

    frontier_states = {initial_state}

    while frontier:
        node = frontier.popleft()
        frontier_states.remove(node.state)
        reached.add(node.state)

        if goal_test(node.state):
            return get_solution(node)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in reached and child_state not in frontier_states:
                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [80]:
def BFS_Kieu_2(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    explored = set()
    frontier_states = {initial_state}

    while frontier:
        node = frontier.popleft()
        frontier_states.remove(node.state)
        explored.add(node.state)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in explored and child_state not in frontier_states:
                if goal_test(child_state):
                    return get_solution(child_node)

                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [81]:
def DFS_Kieu_1(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    reached = set()

    frontier_states = {initial_state}

    while frontier:
        node = frontier.pop()
        frontier_states.remove(node.state)
        reached.add(node.state)

        if goal_test(node.state):
            return get_solution(node)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in reached and child_state not in frontier_states:
                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [82]:
def DFS_Kieu_2(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    explored = set()
    frontier_states = {initial_state}

    while frontier:
        node = frontier.pop()
        frontier_states.remove(node.state)
        explored.add(node.state)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in explored and child_state not in frontier_states:
                if goal_test(child_state):
                    return get_solution(child_node)

                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [83]:
row, col = 4, 4

# Ma trận: 0 (sạch), 1 (bẩn)
mat = [[random.randint(0, 1) for _ in range(col)] for _ in range(row)]
mat_tuple = tuple(tuple(r) for r in mat)

start_x, start_y = 0, 0
while True:
    start_x, start_y = random.randint(0, row-1), random.randint(0, col-1)
    if mat[start_x][start_y] != 2:
        break

initial_state = (start_x, start_y, mat_tuple)

In [84]:
import random
import time
import tkinter as tk
from tkinter import scrolledtext

class VacuumApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Agent Máy Hút Bụi")
        self.root.geometry("900x550")

        self.row, self.col = 4, 4
        self.cell_size = 80

        self.is_animating = False
        self.is_paused = False

        self.setup_ui()
        self.generate_new_environment()

    def setup_ui(self):
        left_frame = tk.Frame(self.root, width=200, bg="#f0f0f0", relief=tk.RAISED, bd=2)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=5, pady=5)

        tk.Label(left_frame, text="BẢNG ĐIỀU KHIỂN", font=("Arial", 12, "bold"), bg="#f0f0f0").pack(pady=10)

        self.algo_var = tk.StringVar(value="BFS1")
        algos = [
            ("BFS Kiểu 1", "BFS1"),
            ("BFS Kiểu 2", "BFS2"),
            ("DFS Kiểu 1", "DFS1"),
            ("DFS Kiểu 2", "DFS2"),
        ]

        for text, val in algos:
            tk.Radiobutton(left_frame, text=text, variable=self.algo_var, value=val, bg="#f0f0f0", font=("Arial", 10)).pack(anchor=tk.W, padx=10, pady=5)

        self.gen_btn = tk.Button(left_frame, text="Tạo Sàn", command=self.generate_new_environment, width=20, bg="#FFD700")
        self.gen_btn.pack(pady=(20, 5))

        self.start_btn = tk.Button(left_frame, text="▶ Chạy Mô Phỏng", command=self.start_simulation, width=20, bg="#32CD32", fg="white", font=("Arial", 10, "bold"))
        self.start_btn.pack(pady=5)

        self.pause_btn = tk.Button(left_frame, text="⏸ Tạm Dừng", command=self.toggle_pause, width=20, bg="#FFA500", fg="white", font=("Arial", 10, "bold"), state=tk.DISABLED)
        self.pause_btn.pack(pady=5)

        self.stop_btn = tk.Button(left_frame, text="⏹ Kết Thúc", command=self.stop_simulation, width=20, bg="#DC143C", fg="white", font=("Arial", 10, "bold"), state=tk.DISABLED)
        self.stop_btn.pack(pady=5)

        bottom_frame = tk.Frame(self.root, height=100, bg="#e6e6e6", relief=tk.SUNKEN, bd=2)
        bottom_frame.pack(side=tk.BOTTOM, fill=tk.X, padx=5, pady=5)

        tk.Label(bottom_frame, text="KẾT QUẢ ĐƯỜNG ĐI:", font=("Arial", 10, "bold"), bg="#e6e6e6").pack(anchor=tk.W, padx=5, pady=(5, 0))
        self.result_text = tk.Text(bottom_frame, height=3, wrap=tk.WORD, font=("Consolas", 10), state=tk.DISABLED)
        self.result_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        right_frame = tk.Frame(self.root, width=250, relief=tk.RAISED, bd=2)
        right_frame.pack(side=tk.RIGHT, fill=tk.Y, padx=5, pady=5)

        tk.Label(right_frame, text="BẢNG LOG CHẠY", font=("Arial", 11, "bold")).pack(pady=5)
        self.log_area = scrolledtext.ScrolledText(right_frame, width=35, wrap=tk.WORD, font=("Consolas", 9))
        self.log_area.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        center_frame = tk.Frame(self.root)
        center_frame.pack(side=tk.LEFT, expand=True)

        self.canvas = tk.Canvas(center_frame, width=self.col*self.cell_size, height=self.row*self.cell_size, bg="white", highlightthickness=2, highlightbackground="black")
        self.canvas.pack(pady=20)

    # CÁC HÀM XỬ LÝ NÚT BẤM
    def toggle_pause(self):
        if not self.is_animating: return
        self.is_paused = not self.is_paused

        if self.is_paused:
            self.pause_btn.config(text="▶ Tiếp Tục", bg="#1E90FF")
            self.log(">>> ĐÃ TẠM DỪNG MÔ PHỎNG.")
        else:
            self.pause_btn.config(text="⏸ Tạm Dừng", bg="#FFA500")
            self.log(">>> TIẾP TỤC MÔ PHỎNG.")

    def stop_simulation(self):
        if not self.is_animating: return

        # Tắt cờ animation để ngắt đứt vòng lặp đệ quy trong animate_step
        self.is_animating = False
        self.is_paused = False

        self.log(">>> ĐÃ HỦY VÀ KẾT THÚC MÔ PHỎNG NGANG CHỪNG.")
        self.reset_button_states()

    def reset_button_states(self):
        self.start_btn.config(state=tk.NORMAL)
        self.gen_btn.config(state=tk.NORMAL)
        self.pause_btn.config(state=tk.DISABLED, text="⏸ Tạm Dừng", bg="#FFA500")
        self.stop_btn.config(state=tk.DISABLED)

    # CÁC HÀM HIỂN THỊ LOG VÀ BẢN ĐỒ
    def log(self, message):
        self.log_area.insert(tk.END, message + "\n")
        self.log_area.see(tk.END)

    def update_result(self, message):
        self.result_text.config(state=tk.NORMAL)
        self.result_text.delete(1.0, tk.END)
        self.result_text.insert(tk.END, message)
        self.result_text.config(state=tk.DISABLED)

    def generate_new_environment(self):
        if self.is_animating: return

        self.mat = [[random.randint(0, 1) for _ in range(self.col)] for _ in range(self.row)]
        while True:
            self.start_x, self.start_y = random.randint(0, self.row-1), random.randint(0, self.col-1)
            if self.mat[self.start_x][self.start_y] != 2:
                break

        self.current_x, self.current_y = self.start_x, self.start_y
        self.current_grid = [list(r) for r in self.mat]

        self.log_area.delete(1.0, tk.END)
        self.log("Đã khởi tạo môi trường mới.")
        self.update_result("Chưa có kết quả.")
        self.draw_grid()

    def draw_grid(self):
        self.canvas.delete("all")
        for r in range(self.row):
            for c in range(self.col):
                x1, y1 = c * self.cell_size, r * self.cell_size
                x2, y2 = x1 + self.cell_size, y1 + self.cell_size

                val = self.current_grid[r][c]
                color = "white" if val == 0 else "#8B4513" if val == 1 else "black"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="gray")

                if val == 1:
                    self.canvas.create_text(x1+self.cell_size/2, y1+self.cell_size/2, text="🗑️", font=("Arial", 24))

                if r == self.current_x and c == self.current_y:
                    padding = 10
                    self.canvas.create_oval(x1+padding, y1+padding, x2-padding, y2-padding, fill="cyan", outline="blue", width=2)
                    self.canvas.create_text(x1+self.cell_size/2, y1+self.cell_size/2, text="🤖", font=("Arial", 20))

    # HÀM CHẠY VÀ ANIMATION
    def start_simulation(self):
        if self.is_animating: return

        self.is_paused = False

        self.current_x, self.current_y = self.start_x, self.start_y
        self.current_grid = [list(r) for r in self.mat]
        self.draw_grid()
        self.log_area.delete(1.0, tk.END)

        initial_state = (self.start_x, self.start_y, tuple(tuple(r) for r in self.mat))
        algo_choice = self.algo_var.get()

        self.log(f"Đang tìm đường đi bằng: {algo_choice}...")
        self.root.update()

        start_time = time.time()

        if algo_choice == "BFS1": path = BFS_Kieu_1(initial_state)
        elif algo_choice == "BFS2": path = BFS_Kieu_2(initial_state)
        elif algo_choice == "DFS1": path = DFS_Kieu_1(initial_state)
        else: path = DFS_Kieu_2(initial_state)

        elapsed = time.time() - start_time

        if isinstance(path, str):
            self.log("Thất bại! Không tìm được đường đi.")
            self.update_result(path)
        else:
            self.log(f"Tìm thấy đường đi! (Thời gian: {elapsed:.4f}s)")
            self.log(f"Số bước cần đi: {len(path)}")
            self.update_result(" -> ".join(path) if path else "Đã sạch sẽ ngay từ đầu, không cần di chuyển!")

            if len(path) > 0:
                self.is_animating = True

                self.start_btn.config(state=tk.DISABLED)
                self.gen_btn.config(state=tk.DISABLED)
                self.pause_btn.config(state=tk.NORMAL)
                self.stop_btn.config(state=tk.NORMAL)

                self.animate_step(path, 0)

    def animate_step(self, path, step_idx):
        # 1. NẾU NÚT KẾT THÚC ĐƯỢC BẤM -> THOÁT NGAY LẬP TỨC
        if not self.is_animating:
            return

        # 2. KIỂM TRA TẠM DỪNG
        if self.is_paused:
            self.root.after(200, self.animate_step, path, step_idx)
            return

        # 3. NẾU ĐÃ CHẠY HẾT ĐƯỜNG ĐI
        if step_idx >= len(path):
            self.is_animating = False
            self.reset_button_states()
            self.log("HOÀN THÀNH DỌN DẸP!")
            return

        # 4. THỰC HIỆN BƯỚC ĐI
        action = path[step_idx]
        self.log(f"Bước {step_idx + 1}: Thực hiện {action}")

        if action == "CLEAN":
            self.current_grid[self.current_x][self.current_y] = 0
        elif action == "UP":    self.current_x -= 1
        elif action == "DOWN":  self.current_x += 1
        elif action == "LEFT":  self.current_y -= 1
        elif action == "RIGHT": self.current_y += 1

        self.draw_grid()

        # 5. CHỜ 0.4s RỒI CHẠY BƯỚC TIẾP
        self.root.after(400, self.animate_step, path, step_idx + 1)

# Chạy giao diện
if __name__ == "__main__":
    try:
        root.destroy()
    except:
        pass

    root = tk.Tk()
    app = VacuumApp(root)
    root.mainloop()